# Data preparation

In [15]:
import json
import random
from pathlib import Path
from utils import load_images

my_random = 2441156
metadata_path = '../../../../Datasets/PlantVillage-Dataset/leaf_grouping/leaf-map.json'

### Get images data
Use prepared utility function based on functions developed in 01_eda.ipynb

In [7]:
images_data = load_images()

images_path = images_data["images_path"]
classes = images_data["classes"]
number_classes = images_data["number_classes"]
class_counts = images_data["class_counts"]
class_distribution = images_data["class_distribution"]
clean_images = images_data["clean_images"]
all_rgb = images_data["all_rgb"]

print(f"Dataset Path: {images_path}")
print(f"Total Classes: {number_classes}")
print(f"Total Clean Images: {len(clean_images)}")
print(f"Are all images valid RGB JPEGs? {all_rgb}")
print(f"Smallest class size: {min(class_counts)}")
print(f"Largest class size: {max(class_counts)}")

Validating image properties (RGB / JPEG)...


100%|██████████| 54304/54304 [00:06<00:00, 8513.09it/s]

Dataset Path: C:\Users\kmiel\Repository\ARU MSc\Datasets\PlantVillage-Dataset\raw\color
Total Classes: 38
Total Clean Images: 54304
Are all images valid RGB JPEGs? True
Smallest class size: 152
Largest class size: 5507


### Import images metadata
Import metadata from leaf-map.json file in repository to inspect it

In [10]:
with open(metadata_path) as metadata_file:
    images_metadata = json.load(metadata_file)

print(f"1. Type: {type(images_metadata)}")

print(f"2. Total top-level entries: {len(images_metadata)}")

# Key and value types of one example entry
sample_key, sample_value = next(iter(images_metadata.items()))

print(f"3. Example key type: {type(sample_key)} (e.g., '{sample_key}')")
print(f"   Example value type: {type(sample_value)}")


1. Type: <class 'dict'>
2. Total top-level entries: 40328
3. Example key type: <class 'str'> (e.g., 'rutg._hl 3733')
   Example value type: <class 'list'>


#### Investigate metadata

- How many keys have exactly one value?
- How many keys have more than one value?
- What is the maximum number of values associated with a single key?

In [11]:
# Number of keys with one value
one_value_count = sum(1 for values in images_metadata.values() if len(values) == 1)

# Number of keys with multiple values
multiple_values_count = sum(1 for values in images_metadata.values() if len(values) > 1)

# Max number of values per key
max_values = max(len(value) for value in images_metadata.values())

print(f"Keys with exactly one value: {one_value_count}")
print(f"Keys with more than one value: {multiple_values_count}")
print(f"Maximum number of values for a single key: {max_values}")

Keys with exactly one value: 38156
Keys with more than one value: 2172
Maximum number of values for a single key: 3


In [12]:
total_individual_mappings = sum(len(values) for values in images_metadata.values())
print(f'Total number of individual mappings: {total_individual_mappings}')

Total number of individual mappings: 42520


#### Extract the image identifier
Image file names follow this pattern: `0b8dabb7-5f1b-4fdc-b3fa-30b289707b90___JR_FrgE.S 3047.JPG` while the metadata resembles `jr_frge.s 3047`.  
The identifier from the image files must be extracted in order to lookup the metadata in the json file.

In [ ]:
# helper function to extract image identifier names
def get_image_id(image_path: Path):
    # remove the file extension
    stem = image_path.stem
    # split by '___'

    identifier = stem.split("___")[1]

    # standardise to lower case as this is what leaf-map.json uses
    return identifier.lower()

Inspect how the directory, file and metadata naming relates to each other using 3 examples

In [ ]:
def split_metadata(mapping):
    return mapping.split(":::")

In [51]:
random.seed(my_random)

images_to_check = random.sample(clean_images, 3)

print(f'Number of images to check: {len(images_to_check)}\n')

for img_path in images_to_check:
    image_id = get_image_id(img_path)

    mappings = {}
    
    mapping = images_metadata.get(image_id)
    
    print(f'Class directory: {img_path.parent.name}')
    print(f'Filename: {img_path.name}')
    print(f'Image ID: {image_id}')
    print(f'Mapping: {mapping}')

    for item in mapping:
        split = split_metadata(item)
        mappings[split[0]] = split[1]
        print(f'Split mapping:\n    Leaf class: {split[0]}\n    Leaf group: {split[1]}\n')




Number of images to check: 3

Class directory: Soybean___healthy
Filename: 637c4131-9177-4e3c-88fa-7fcacbca33ba___RS_HL 4655.JPG
Image ID: rs_hl 4655
Mapping: ['Soybean___healthy:::218.0']
Split mapping:
    Leaf class: Soybean___healthy
    Leaf group: 218.0

Class directory: Tomato___Leaf_Mold
Filename: 790cfb1d-4926-46bb-bd50-b750e823c72a___Crnl_L.Mold 7130.JPG
Image ID: crnl_l.mold 7130
Mapping: ['Tomato___Leaf_Mold:::121.0']
Split mapping:
    Leaf class: Tomato___Leaf_Mold
    Leaf group: 121.0

Class directory: Tomato___Bacterial_spot
Filename: 61a05330-1086-4891-8482-3f422ff59492___GCREC_Bact.Sp 3429.JPG
Image ID: gcrec_bact.sp 3429
Mapping: ['Tomato___Bacterial_spot:::121.0']
Split mapping:
    Leaf class: Tomato___Bacterial_spot
    Leaf group: 121.0

